# 2.5 Automatic Differentiation

Computing derivatives by hand is tedious and error-prone, especially once a model has thousands of parameters. Deep learning frameworks solve this with **automatic differentiation** (autograd): as the forward computation runs, PyTorch builds a **computational graph** that records which tensors combine to produce which others; calling `.backward()` then walks that graph *backwards*, applying the **chain rule** at every recorded operation. This process is called **backpropagation**.

This section works through the mechanics of `torch.autograd` on small, self-contained examples: differentiating a single vector function, handling non-scalar outputs, detaching part of a graph from gradient tracking, and computing gradients through ordinary Python control flow (loops and `if` statements).

In [1]:
import torch

## 2.5.1 A Simple Function

As a toy example, suppose we are interested in differentiating $y = 2\mathbf{x}^\top\mathbf{x}$ with respect to the column vector $\mathbf{x}$. We first create `x` and give it some initial values.

In [2]:
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

Before computing any gradient we need a place to store it. It matters that we do not allocate new memory every time we differentiate with respect to a parameter, since in a real model we might update the same parameters thousands or millions of times and would quickly exhaust memory. Calling `requires_grad_(True)` tells PyTorch to track every operation performed on `x` and reserves a `.grad` attribute for the result — `x.grad` is `None` until a backward pass actually fills it in.

In [3]:
# Can also create x = torch.arange(4.0, requires_grad=True)
x.requires_grad_(True)
x.grad # The gradient is None by default

Now we compute $y = 2\mathbf{x}^\top\mathbf{x}$. Because `x` has `requires_grad=True`, PyTorch records this operation (and every tensor operation that touches `x`) into a computational graph as it runs — that is why `y` prints with a `grad_fn`, a pointer back to the operation that produced it, which is what `.backward()` will use to walk the graph in reverse.

In [4]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

Calling `y.backward()` triggers autograd: starting at `y`, it walks the recorded graph backwards, applying the chain rule at each step, and writes the result into `x.grad`. For $y = 2\mathbf{x}^\top\mathbf{x}$ the gradient is $\frac{\partial y}{\partial \mathbf{x}} = 4\mathbf{x}$ — we can check that directly against the values in `x`.

In [5]:
# Compute gradients
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

In [6]:
x.grad == 4 * x   # verify against the known closed-form gradient of y = 2*x^T*x

tensor([True, True, True, True])

### Gradients accumulate by default

PyTorch does not overwrite `x.grad` on the next `backward()` call — it **adds** the newly computed gradient to whatever is already stored there. That default is convenient when we deliberately want to accumulate gradients across several backward passes (for example, summing over the minibatches of one large batch), but it silently corrupts the result if we forget about it. The cell below computes a *different* function, $y_2 = \sum_i x_i$ (gradient $\mathbf{1}$), and calls `backward()` again without resetting `x.grad` first — watch the old gradient $4\mathbf{x}$ get added into the new one instead of being replaced by it.

In [7]:
y2 = x.sum()   # a different function this time: dy2/dx_i = 1 for every i
y2.backward()  # backward() ADDS into x.grad instead of overwriting it
x.grad         # [0,4,8,12] (still there from y) + [1,1,1,1] (new) = [1,5,9,13]

tensor([ 1.,  5.,  9., 13.])

This is exactly why the idiom below matters: call `x.grad.zero_()` before every fresh `backward()` unless the accumulation itself is what you want. Repeating the same $y = \sum_i x_i$ computation, but zeroing first, now gives the correct, unmixed gradient $\mathbf{1}$.

In [8]:
x.grad.zero_()   # reset the gradient buffer before this fresh backward pass
y = x.sum()      # y = sum(x_i), so dy/dx_i = 1 for every i
y.backward()
x.grad

tensor([1., 1., 1., 1.])

## 2.5.2 Backward for Non-Scalar Variables

When `y` is a vector rather than a scalar, the natural derivative of `y` with respect to a vector `x` is a matrix — the **Jacobian**, holding every component of `y`'s partial derivative with respect to every component of `x`. In deep learning we rarely want that full matrix: far more often `y` holds a per-example loss for a batch, and we just want the *summed* gradient across examples, a vector shaped like `x`. Because frameworks differ on how to interpret gradients of non-scalar tensors, PyTorch refuses to guess: calling `.backward()` on a non-scalar `y` raises an error unless we supply a `gradient` argument — a vector `v` shaped like `y` — telling it to compute the **vector-Jacobian product** $\mathbf{v}^\top \partial y/\partial x$ instead of the full Jacobian. Passing a vector of ones has the same effect as summing `y` into a scalar first and then calling `.backward()` — and summing first is usually the faster route.

In [9]:
# 1. Clear previous gradients. 
# PyTorch accumulates gradients by default; if we don't zero them out, 
# the new results will be added to whatever was stored in x.grad previously.
if x.grad is not None:
    x.grad.zero_()

# 2. Define the Forward Pass.
# We are creating the function y = f(x) = x^2.
y = x * x

print(y)

# 3. Perform the Backward Pass (Backpropagation).
# Since 'y' is a vector (tensor) and not a single scalar (like a loss value), 
# we must provide a 'gradient' argument of the same shape. 
# Passing torch.ones() essentially means dL/dy = 1 for each element.
# Mathematically, this computes dy/dx = 2*x.
y.backward(gradient=torch.ones(len(y))) 

# Faster equivalent: y.sum().backward() -- sums y into a scalar first,
# then backpropagates through that; same gradient, fewer ops.

# 4. Access the result.
# x.grad now holds the values of the derivative evaluated at the points in x.
x.grad

tensor([0., 1., 4., 9.], grad_fn=<MulBackward0>)


tensor([0., 2., 4., 6.])

## 2.5.3 Detaching Computation

Sometimes we want to move part of a computation outside the recorded graph — for instance, to build an auxiliary term from `x` that we do *not* want contributing a gradient. Concretely: suppose $z = x \cdot y$ where $y = x^2$, so that $z = x^3$, but we only care about $x$'s *direct* effect on $z$, not the effect that flows through $y$. `u = y.detach()` gives us a tensor `u` with $y$'s values but no graph history — its provenance is wiped out, so gradients cannot flow through it back to `x`. Computing $z = u \cdot x$ and backpropagating then treats `u` as a plain constant: the gradient comes out to $u$ itself, not the $3x^2$ we would get from differentiating $x^3$ directly.

In [10]:
# Reset the gradients for the tensor x to zero to prevent accumulation from previous steps
x.grad.zero_()

# Compute y = x^2; y is part of the computational graph and tracks gradients
y = x * x

# Create a new tensor 'u' that has the same value as 'y' but is "detached" 
# from the gradient history. PyTorch won't backpropagate through u to x.
u = y.detach()

# Compute z = u * x. Since u is treated as a constant, 
# the derivative dz/dx is simply u.
z = u * x

# Sum all elements and compute gradients. 
# Backpropagation starts here, but stops at 'u'.
z.sum().backward()

# Verification: 
# x.grad should be equal to u because z = u * x (treating u as a constant).
# This returns (x, True, u, x.grad) assuming the values match.
x, x.grad == u, u, x.grad


(tensor([0., 1., 2., 3.], requires_grad=True),
 tensor([True, True, True, True]),
 tensor([0., 1., 4., 9.]),
 tensor([0., 1., 4., 9.]))

Detaching `u` only severed the path through `u` — the graph leading to `y = x * x` itself is untouched, so we can still differentiate `y` with respect to `x` directly. Zeroing the gradient and backpropagating through `y.sum()` recovers the ordinary result $\partial y/\partial x = 2\mathbf{x}$, confirming `detach()` only cut the one path we asked it to.

In [11]:
# Clear the previous gradients stored in x to ensure a fresh calculation
x.grad.zero_()

# Perform backpropagation on y = x * x
# Since y is still connected to x in the graph, the derivative is dy/dx = 2x
y.sum().backward()

# Verification:
# This will return True (or a tensor of True values) because 
# the gradient of x^2 is indeed 2*x.
x.grad == 2 * x

tensor([True, True, True, True])

## 2.5.4 Gradients and Python Control Flow

One benefit of a **dynamic**, define-by-run autograd system like PyTorch's is that it handles ordinary Python control flow without any special casing — loops whose length depends on the data, or `if` statements that branch on a tensor's value. PyTorch simply records whatever sequence of operations actually executes for a given input, and backpropagates through that specific trace. The function below doubles its input repeatedly until the result's norm passes 1000, then scales it once more depending on the sign of the sum; both the number of loop iterations and the branch taken depend on the random value of `a`.

In [12]:
def f(a):
    # Initial operation: b is twice a
    b = a * 2
    
    # Dynamic Loop: The number of iterations depends on the value of 'b'.
    # PyTorch records each multiplication in the graph during execution.
    while b.norm() < 1000:
        b = b * 2
        
    # Control Flow: The path taken depends on the data. 
    # Autograd records which branch was taken (the 'if' or the 'else').
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
        
    return c

For any particular input `a`, `f` is a **linear function of `a` with piecewise-defined scale**: whichever trace actually executes is built entirely out of scalar multiplications, so it computes $d = f(a) = k \cdot a$ for some scale $k$ — fixed by the loop count and branch taken for that trace, but constant with respect to `a`'s value within it. Because $d$ is linear in `a` along the executed trace, $d/a$ must equal the gradient $\partial d/\partial a$; we can verify that directly.

In [13]:
# Create a random scalar tensor 'a' and enable gradient tracking
a = torch.randn(size=(), requires_grad=True)

# Pass 'a' through the dynamic function. 
# PyTorch builds the graph on-the-fly based on how many 
# loop iterations occur and which 'if' branch is taken.
d = f(a)

# Backpropagate to calculate the gradient da/dd
d.backward()

In [14]:
a.grad == d / a  # f is linear in a along the executed trace (d = k*a), so da/dd = k = d/a

tensor(True)

## 2.5.5 Summary

- **Autograd** builds a computational graph as the forward pass runs, tracking how each value depends on others; `.backward()` walks that graph backwards applying the chain rule — the algorithm known as **backpropagation**.
- The basic recipe: (i) attach gradients to the variables we want derivatives for, via `requires_grad_(True)`; (ii) record the computation that produces the target value; (iii) call `.backward()`; (iv) read off the result from `.grad`.
- **Gradients accumulate**: PyTorch adds each new `backward()` result into `.grad` rather than overwriting it — convenient when we deliberately want to sum gradients across several passes, but call `x.grad.zero_()` first whenever we don't.
- `.backward()` needs a scalar. For a non-scalar `y`, either sum it first (`y.sum().backward()`, usually faster) or pass an explicit `gradient` argument shaped like `y` — the vector-Jacobian product this computes is what makes the two agree.
- `.detach()` returns a tensor with the same values but no graph history, letting part of a computation be treated as a constant during backpropagation, while leaving the rest of the graph fully differentiable.
- Because PyTorch traces whatever code actually executes, gradients flow correctly through ordinary Python loops and `if` statements — it is the trace realized for a given input, not the source code in the abstract, that gets differentiated.